# Project 07: Burrows-Wheeler Transform

This notebook implements the Burrows-Wheeler Transform (BWT), suffix arrays,
FM-index backward search, and run-length encoding. The implementation uses
uppercase text and `$` as a unique end-of-string sentinel.

The occurrence table uses **inclusive indexing**: `occur[c][i]` is the number
of occurrences of `c` in `bwt[:i + 1]`.

In [ ]:
from collections import Counter
from itertools import groupby

SENTINEL = '$'


def normalize_text(text: str, sentinel: str = SENTINEL) -> str:
    """Normalize text and ensure that the sentinel occurs exactly once."""
    if not isinstance(text, str):
        raise TypeError('text must be a string')

    text = text.upper()
    if sentinel in text[:-1]:
        raise ValueError(
            f'{sentinel!r} must not occur inside the input text.'
        )
    return text if text.endswith(sentinel) else text + sentinel


def BWT(text: str) -> str:
    """Compute the BWT using the educational rotation-matrix method."""
    text = normalize_text(text)
    rotations = [text[i:] + text[:i] for i in range(len(text))]
    return ''.join(rotation[-1] for rotation in sorted(rotations))


def suffix_array(text: str) -> list[int]:
    """Return suffix starting positions in lexicographic order."""
    text = normalize_text(text)
    return sorted(range(len(text)), key=lambda index: text[index:])


def BWT_from_suffix_array(text: str, suffix_positions: list[int]) -> str:
    """Compute a BWT from text and a complete suffix array."""
    if len(suffix_positions) != len(text):
        raise ValueError('The suffix array must contain one position per character.')
    if sorted(suffix_positions) != list(range(len(text))):
        raise ValueError('suffix_positions must be a permutation of text positions.')

    return ''.join(text[pos - 1] if pos else text[-1] for pos in suffix_positions)


def cal_count(bwt_string: str) -> dict[str, int]:
    """Map each character to the number of lexicographically smaller symbols."""
    frequencies = Counter(bwt_string)
    counts = {}
    total = 0
    for char in sorted(frequencies):
        counts[char] = total
        total += frequencies[char]
    return counts


def cal_occur(bwt_string: str) -> dict[str, list[int]]:
    """Build an inclusive occurrence table for a BWT string.

    `occur[char][i]` equals the number of `char` symbols in
    `bwt_string[:i + 1]`.
    """
    alphabet = sorted(set(bwt_string))
    occur = {char: [0] * len(bwt_string) for char in alphabet}
    running = {char: 0 for char in alphabet}

    for index, current in enumerate(bwt_string):
        running[current] += 1
        for char in alphabet:
            occur[char][index] = running[char]
    return occur


def update_range(lower: int, upper: int, count: dict[str, int],
                 occur: dict[str, list[int]], char: str) -> tuple[int, int]:
    """Update an inclusive suffix-array range during backward search."""
    if char not in count:
        return 1, 0

    new_lower = count[char] if lower == 0 else count[char] + occur[char][lower - 1]
    new_upper = count[char] + occur[char][upper] - 1
    return new_lower, new_upper


def find_match(query: str, reference: str) -> list[int]:
    """Return all zero-based positions where query occurs in reference."""
    if not isinstance(query, str) or not isinstance(reference, str):
        raise TypeError('query and reference must be strings')

    query = query.upper()
    reference = normalize_text(reference)
    if not query or SENTINEL in query:
        return []

    suffix_positions = suffix_array(reference)
    bwt = BWT_from_suffix_array(reference, suffix_positions)
    count = cal_count(bwt)
    occur = cal_occur(bwt)

    lower, upper = 0, len(bwt) - 1
    for char in reversed(query):
        lower, upper = update_range(lower, upper, count, occur, char)
        if lower > upper:
            return []

    text_length = len(reference) - 1
    return sorted(
        position for position in suffix_positions[lower:upper + 1]
        if position + len(query) <= text_length
    )


def run_length_encode(text: str) -> list[tuple[str, int]]:
    """Represent consecutive runs as `(character, count)` tuples."""
    return [(char, sum(1 for _ in group)) for char, group in groupby(text)]


def run_length_decode(encoded: list[tuple[str, int]]) -> str:
    """Decode the tuple representation produced by run_length_encode."""
    return ''.join(char * count for char, count in encoded)


## Examples and tests

The assertions below provide a small regression suite and can be rerun after
any implementation change.

In [ ]:
assert BWT('banana') == 'ANNB$AA'
assert BWT('googol') == 'LO$OOGG'
assert suffix_array('banana') == [6, 5, 3, 1, 0, 4, 2]
assert BWT_from_suffix_array('banana$', suffix_array('banana$')) == 'annb$aa'

assert cal_count('banana') == {'a': 0, 'b': 3, 'n': 4}
assert cal_occur('annb$aa') == {
    '$': [0, 0, 0, 0, 1, 1, 1],
    'a': [1, 1, 1, 1, 1, 2, 3],
    'b': [0, 0, 0, 1, 1, 1, 1],
    'n': [0, 1, 2, 2, 2, 2, 2],
}

assert find_match('ana', 'banana') == [1, 3]
assert find_match('na', 'banana') == [2, 4]
assert find_match('xyz', 'banana') == []
assert find_match('ANA', 'banana') == [1, 3]

encoded = run_length_encode('aaabbc')
assert encoded == [('a', 3), ('b', 2), ('c', 1)]
assert run_length_decode(encoded) == 'aaabbc'

print('All tests passed.')